# arXiv Computer Science 샘플 200개 (PDF 포함) 수집

최근 3개월 이내에 제출된 Computer Science(`cs.*`) 논문 중 최신 200편의 메타데이터와 PDF 원문 파일을 함께 수집합니다.

> 메타데이터는 `data/pdf_200/arxiv_cs_pdf_200.jsonl`에, PDF 파일은 `data/pdf_200/pdfs/{arxiv_id}.pdf`에 저장됩니다.
> 이미 받아둔 PDF는 재실행 시 다시 내려받지 않고 건너뜁니다.

In [1]:
from calendar import monthrange
from datetime import date
from pathlib import Path
import http.client
import json
import ssl
import certifi
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

MONTHS = 3
SAMPLE_SIZE = 200
PAGE_SIZE = 50
REQUEST_INTERVAL_SECONDS = 10.0
MAX_RETRIES = 10
BACKOFF_BASE_SECONDS = 5.0
BACKOFF_MAX_SECONDS = 180.0
API_URL = 'https://export.arxiv.org/api/query'
USER_AGENT = 'arxiv-cs-pdf-200/1.0'
SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())

def subtract_months(day, months):
    """월말에서도 안전하게 지정한 개월 수를 뺍니다."""
    month_index = day.year * 12 + day.month - 1 - months
    year, month_zero_based = divmod(month_index, 12)
    month = month_zero_based + 1
    return date(year, month, min(day.day, monthrange(year, month)[1]))

END_DATE = date.today()
START_DATE = subtract_months(END_DATE, MONTHS)
DATE_QUERY = f'submittedDate:[{START_DATE:%Y%m%d}0000 TO {END_DATE:%Y%m%d}2359]'
SEARCH_QUERY = f'cat:cs.* AND {DATE_QUERY}'

# 노트북을 어디서 실행해도 프로젝트 루트의 data 폴더에 저장
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'pdf_200'
PDF_DIR = OUTPUT_DIR / 'pdfs'
JSONL_PATH = OUTPUT_DIR / 'arxiv_cs_pdf_200.jsonl'
PDF_DIR.mkdir(parents=True, exist_ok=True)

print(f'조회 기간: {START_DATE} ~ {END_DATE}')
print(f'검색식: {SEARCH_QUERY}')
print(f'샘플 크기: {SAMPLE_SIZE}건')
print(f'PDF 저장 위치: {PDF_DIR.resolve()}')
print(f'메타데이터 저장 위치: {JSONL_PATH.resolve()}')

조회 기간: 2026-06-14 ~ 2026-09-14
검색식: cat:cs.* AND submittedDate:[202606140000 TO 202609142359]
샘플 크기: 200건
PDF 저장 위치: C:\Users\Playdata\Desktop\arxiv_graph_RAG\data\pdf_200\pdfs
메타데이터 저장 위치: C:\Users\Playdata\Desktop\arxiv_graph_RAG\data\pdf_200\arxiv_cs_pdf_200.jsonl


In [2]:
ATOM_NS = {'atom': 'http://www.w3.org/2005/Atom'}
ARXIV_NS = {'arxiv': 'http://arxiv.org/schemas/atom'}
OPENSEARCH_NS = {'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'}

def text_or_none(parent, path, namespaces=ATOM_NS):
    node = parent.find(path, namespaces)
    return node.text.strip() if node is not None and node.text else None

def parse_entry(entry):
    authors = []
    for author in entry.findall('atom:author', ATOM_NS):
        name = text_or_none(author, 'atom:name')
        if name:
            authors.append(name)
    categories = [
        node.attrib['term']
        for node in entry.findall('atom:category', ATOM_NS)
        if 'term' in node.attrib
    ]
    links = {
        link.attrib.get('rel', 'alternate'): link.attrib.get('href')
        for link in entry.findall('atom:link', ATOM_NS)
    }
    primary_category_node = entry.find('arxiv:primary_category', ARXIV_NS)
    primary_category = primary_category_node.attrib.get('term') if primary_category_node is not None else None
    return {
        'id': text_or_none(entry, 'atom:id'),
        'title': ' '.join((text_or_none(entry, 'atom:title') or '').split()),
        'abstract': ' '.join((text_or_none(entry, 'atom:summary') or '').split()),
        'authors': authors,
        'categories': categories,
        'primary_category': primary_category,
        'published': text_or_none(entry, 'atom:published'),
        'updated': text_or_none(entry, 'atom:updated'),
        'doi': text_or_none(entry, 'arxiv:doi', ARXIV_NS),
        'pdf_url': links.get('related') or links.get('alternate'),
        'source': 'arxiv',
        'collection_window': {
            'start': START_DATE.isoformat(),
            'end': END_DATE.isoformat(),
        },
    }

def _retry_delay(error, attempt):
    retry_after = getattr(error, 'headers', None) and error.headers.get('Retry-After')
    if retry_after:
        try:
            return max(float(retry_after), REQUEST_INTERVAL_SECONDS)
        except ValueError:
            pass
    return min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)

_last_request_at = 0.0
RETRYABLE_HTTP_CODES = {429, 500, 502, 503, 504}
RETRYABLE_NETWORK_ERRORS = (urllib.error.URLError, TimeoutError, ConnectionError, http.client.HTTPException)

def _urlopen_with_retry(request, timeout):
    global _last_request_at
    for attempt in range(MAX_RETRIES + 1):
        elapsed = time.monotonic() - _last_request_at
        if elapsed < REQUEST_INTERVAL_SECONDS:
            time.sleep(REQUEST_INTERVAL_SECONDS - elapsed)
        try:
            _last_request_at = time.monotonic()
            return urllib.request.urlopen(request, timeout=timeout, context=SSL_CONTEXT)
        except urllib.error.HTTPError as error:
            if error.code not in RETRYABLE_HTTP_CODES or attempt >= MAX_RETRIES:
                raise
            delay = _retry_delay(error, attempt)
            print(f'HTTP {error.code}: {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})')
            error.close()
            time.sleep(delay)
        except RETRYABLE_NETWORK_ERRORS as error:
            if attempt >= MAX_RETRIES:
                raise
            delay = min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)
            print(f'네트워크 오류({error!r}): {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})')
            time.sleep(delay)

def fetch_feed(query, start=0, max_results=PAGE_SIZE):
    """검색식으로 한 페이지를 조회하고 (전체 결과 수, 이 페이지의 논문 목록)을 반환합니다."""
    params = {
        'search_query': query,
        'start': start,
        'max_results': max_results,
        'sortBy': 'submittedDate',
        'sortOrder': 'descending',
    }
    url = f'{API_URL}?{urllib.parse.urlencode(params)}'
    request = urllib.request.Request(url, headers={'User-Agent': USER_AGENT})
    with _urlopen_with_retry(request, timeout=60) as response:
        root = ET.fromstring(response.read())
    total_node = root.find('opensearch:totalResults', OPENSEARCH_NS)
    total_results = int(total_node.text) if total_node is not None and total_node.text else 0
    entries = [parse_entry(entry) for entry in root.findall('atom:entry', ATOM_NS)]
    return total_results, entries

def pdf_filename(paper_id):
    return paper_id.rstrip('/').rsplit('/', 1)[-1] + '.pdf'

def download_pdf(pdf_url, destination):
    if destination.exists() and destination.stat().st_size > 0:
        return 'cached'
    request = urllib.request.Request(pdf_url, headers={'User-Agent': USER_AGENT})
    with _urlopen_with_retry(request, timeout=120) as response:
        content = response.read()
    if not content.startswith(b'%PDF'):
        raise ValueError('응답이 PDF 파일이 아닙니다.')
    destination.write_bytes(content)
    return 'downloaded'

In [3]:
def write_records(records):
    with JSONL_PATH.open('w', encoding='utf-8') as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + '\n')

print('메타데이터 수집 중...')
papers = []
seen_ids = set()
for start in range(0, SAMPLE_SIZE, PAGE_SIZE):
    _, page = fetch_feed(SEARCH_QUERY, start=start, max_results=min(PAGE_SIZE, SAMPLE_SIZE - start))
    if not page:
        break
    for paper in page:
        if paper['id'] and paper['id'] not in seen_ids:
            seen_ids.add(paper['id'])
            papers.append(paper)
    print(f'  {len(papers)}개 메타데이터 누적 수집')
    if len(papers) >= SAMPLE_SIZE:
        break

papers = papers[:SAMPLE_SIZE]
print(f'메타데이터 수집 완료: {len(papers)}개')

records = []
for index, paper in enumerate(papers, start=1):
    record = dict(paper)
    pdf_path = PDF_DIR / pdf_filename(paper['id'])
    record['pdf_local_path'] = str(pdf_path)
    try:
        record['pdf_status'] = download_pdf(paper['pdf_url'], pdf_path)
        record['pdf_error'] = None
    except Exception as error:
        record['pdf_status'] = 'error'
        record['pdf_error'] = f'{type(error).__name__}: {error}'
    records.append(record)
    write_records(records)
    print(f'[{index}/{len(papers)}] {record["pdf_status"]}: {record["title"][:70]}')

success_count = sum(record['pdf_status'] in ('downloaded', 'cached') for record in records)
print(f'완료: {len(records)}개 레코드 저장, PDF 확보 {success_count}/{len(records)}개')
print(f'메타데이터: {JSONL_PATH.resolve()}')
print(f'PDF 파일: {PDF_DIR.resolve()}')

메타데이터 수집 중...
  50개 메타데이터 누적 수집
HTTP 429: 5초 후 재시도 (1/10)
HTTP 429: 10초 후 재시도 (2/10)


KeyboardInterrupt: 

In [ ]:
# JSONL 저장 결과 및 PDF 파일 검증
with JSONL_PATH.open(encoding='utf-8') as file:
    saved_records = [json.loads(line) for line in file if line.strip()]

assert len(saved_records) == len(records)
assert len({record['id'] for record in saved_records}) == len(saved_records)
assert all(record['id'] and record['title'] and record['abstract'] for record in saved_records)
assert all(START_DATE.isoformat() <= record['published'][:10] <= END_DATE.isoformat() for record in saved_records)

downloaded_pdfs = list(PDF_DIR.glob('*.pdf'))
print(f'검증 완료: {len(saved_records)}개 레코드, 중복 없음, 최근 {MONTHS}개월 범위 확인')
print(f'실제 저장된 PDF 파일 수: {len(downloaded_pdfs)}개')
display(saved_records[:2])